# AI-Based Job Market Skill Gap & Future Trend Analysis - V2

This notebook analyzes job datasets to answer:

1. What skills are most requested in job offers?
2. What job categories appear most often?
3. Which skills are trending upward?
4. What skills are missing for a target job?
5. What skills should a candidate learn next?

It is designed to work with the uploaded files:

- `data_jobs.xlsx`
- `indeed-jobs-data.xlsx`

In [ ]:
# =========================
# 1. Import libraries
# =========================

import pandas as pd
import numpy as np
import re
import ast
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
# =========================
# 2. Load datasets
# =========================

data_jobs = pd.read_excel("data_jobs.xlsx")
indeed_jobs = pd.read_excel("indeed-jobs-data.xlsx")

print("data_jobs columns:", data_jobs.columns.tolist())
print("indeed_jobs columns:", indeed_jobs.columns.tolist())

# Dataset 1: data_jobs.xlsx
data_jobs["job_description"] = (
    "Job title: " + data_jobs["job_title"].astype(str) + "\n" +
    "Short title: " + data_jobs["job_title_short"].astype(str) + "\n" +
    "Skills: " + data_jobs["job_skills"].astype(str) + "\n" +
    "Type skills: " + data_jobs["job_type_skills"].astype(str)
)

data_jobs_clean = pd.DataFrame({
    "job_description": data_jobs["job_description"],
    "job_title": data_jobs["job_title"],
    "job_title_short": data_jobs["job_title_short"],
    "job_skills": data_jobs["job_skills"],
    "job_type_skills": data_jobs["job_type_skills"],
    "job_posted_date": data_jobs["job_posted_date"],
    "company_name": data_jobs["company_name"],
    "job_location": data_jobs["job_location"],
    "source_file": "data_jobs.xlsx"
})

# Dataset 2: indeed-jobs-data.xlsx
# Dataset 2: indeed-jobs-data.xlsx
indeed_jobs["job_description"] = (
    "Job title: " + indeed_jobs["positionName"].astype(str) + "\n" +
    "Search position: " + indeed_jobs["searchInput/position"].astype(str) + "\n" +
    "Description: " + indeed_jobs["description"].astype(str)
)

indeed_jobs_clean = pd.DataFrame({
    "job_description": indeed_jobs["job_description"],
    "job_title": indeed_jobs["positionName"],
    "job_title_short": indeed_jobs["searchInput/position"],
    "job_skills": "",
    "job_type_skills": "",
    "job_posted_date": indeed_jobs["postingDateParsed"],
    "company_name": indeed_jobs["company"],
    "job_location": indeed_jobs["location"],
    "source_file": "indeed-jobs-data.xlsx"
})

# Combine both datasets
df = pd.concat([data_jobs_clean, indeed_jobs_clean], ignore_index=True)

df["job_description"] = df["job_description"].astype(str).str.strip()
df = df[df["job_description"] != ""].reset_index(drop=True)

print(f"✅ {len(df)} job descriptions loaded from both datasets")
display(df.head())

In [ ]:
# =========================
# 4. Standardize dataset
# =========================

jobs = pd.DataFrame()

jobs["job_title"] = df["job_title"].astype(str)
jobs["short_title"] = df["job_title_short"].astype(str)
jobs["description"] = df["job_description"].astype(str)
jobs["raw_skills"] = df["job_skills"].astype(str)
jobs["raw_type_skills"] = df["job_type_skills"].astype(str)

jobs["posted_date"] = pd.to_datetime(
    df["job_posted_date"],
    errors="coerce",
    utc=True
).dt.tz_localize(None)

jobs["company"] = df["company_name"].astype(str)
jobs["location"] = df["job_location"].astype(str)
jobs["source_file"] = df["source_file"].astype(str)

jobs["text"] = (
    jobs["job_title"].fillna("") + " " +
    jobs["description"].fillna("") + " " +
    jobs["raw_skills"].fillna("") + " " +
    jobs["raw_type_skills"].fillna("")
)

jobs = jobs.drop_duplicates(
    subset=["job_title", "description", "raw_skills"]
).reset_index(drop=True)

print("Final standardized dataset shape:", jobs.shape)
display(jobs.head())

In [ ]:
# =========================
# 5. Clean text
# =========================

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)      # remove html tags
    text = re.sub(r"[^a-zA-Z0-9+#.\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

jobs["clean_text"] = jobs["text"].apply(clean_text)
jobs[["job_title", "clean_text"]].head()

In [ ]:
# =========================
# 6. Skill dictionary
# =========================

skill_categories = {
    "Programming": [
        "python", "java", "javascript", "typescript", "c++", "c#", "php",
        "ruby", "go", "golang", "scala", "r", "bash", "powershell", "sql"
    ],
    "Frontend": [
        "html", "css", "react", "angular", "vue", "next.js", "tailwind", "bootstrap"
    ],
    "Backend": [
        "spring boot", "node.js", "express", "fastapi", "django", "flask", ".net",
        "rest api", "graphql", "microservices"
    ],
    "Data": [
        "excel", "power bi", "tableau", "pandas", "numpy", "statistics",
        "data analysis", "data analytics", "etl", "data visualization",
        "machine learning", "deep learning", "ai", "artificial intelligence",
        "nlp", "big data", "spark", "hadoop"
    ],
    "Cloud DevOps": [
        "aws", "azure", "gcp", "google cloud", "docker", "kubernetes",
        "terraform", "jenkins", "gitlab ci", "github actions", "ci/cd",
        "linux", "devops"
    ],
    "Database": [
        "postgresql", "mysql", "oracle", "sql server", "mongodb", "redis",
        "snowflake", "bigquery"
    ],
    "Cybersecurity": [
        "cybersecurity", "security", "iam", "soc", "siem", "splunk",
        "firewall", "penetration testing", "vulnerability"
    ],
    "Business": [
        "business analysis", "project management", "agile", "scrum",
        "jira", "stakeholder", "requirements", "kpi", "reporting",
        "communication"
    ]
}

all_skills = sorted(set(skill for skills in skill_categories.values() for skill in skills), key=len, reverse=True)

def extract_skills(text):
    text = clean_text(text)
    found = []
    for skill in all_skills:
        pattern = r"(?<![a-zA-Z0-9+#.])" + re.escape(skill.lower()) + r"(?![a-zA-Z0-9+#.])"
        if re.search(pattern, text):
            found.append(skill)
    return sorted(set(found))

jobs["extracted_skills"] = jobs["clean_text"].apply(extract_skills)
jobs["skill_count"] = jobs["extracted_skills"].apply(len)

display(jobs[["job_title", "extracted_skills", "skill_count"]].head(10))

In [ ]:
# =========================
# 7. Most demanded skills
# =========================

skill_counter = Counter()

for skills in jobs["extracted_skills"]:
    skill_counter.update(skills)

skill_freq = pd.DataFrame(skill_counter.items(), columns=["skill", "frequency"])
skill_freq = skill_freq.sort_values("frequency", ascending=False).reset_index(drop=True)

display(skill_freq.head(30))

plt.figure(figsize=(12, 7))
top_skills = skill_freq.head(20).sort_values("frequency")
plt.barh(top_skills["skill"], top_skills["frequency"])
plt.title("Top 20 Most Requested Skills")
plt.xlabel("Number of Job Posts")
plt.ylabel("Skill")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 8. Detect job category using skills
# =========================

def detect_category(skills):
    scores = {}
    skills_set = set(skills)

    for category, category_skills in skill_categories.items():
        scores[category] = len(skills_set.intersection(set(category_skills)))

    best_category = max(scores, key=scores.get)

    if scores[best_category] == 0:
        title_text = " ".join(skills).lower()
        return "Other"

    return best_category

jobs["job_category"] = jobs["extracted_skills"].apply(detect_category)

category_counts = jobs["job_category"].value_counts().reset_index()
category_counts.columns = ["job_category", "count"]

display(category_counts)

plt.figure(figsize=(10, 6))
plot_data = category_counts.sort_values("count")
plt.barh(plot_data["job_category"], plot_data["count"])
plt.title("Job Categories Distribution")
plt.xlabel("Number of Job Posts")
plt.ylabel("Job Category")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 9. ML model: predict job category from job text
# =========================

model_data = jobs[jobs["job_category"] != "Other"].copy()

if model_data["job_category"].nunique() >= 2 and len(model_data) >= 20:
    X = model_data["clean_text"]
    y = model_data["job_category"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english")
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train_vec, y_train)

    y_pred = clf.predict(X_test_vec)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))
else:
    print("Not enough labeled data to train a reliable model.")

## Advanced Model Evaluation

This section adds stronger evaluation methods for the machine learning model:

- Confusion Matrix
- Cross Validation
- Model Comparison
- Feature importance for skills/keywords

In [ ]:
# =========================
# 9.1 Confusion Matrix
# =========================

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

if "clf" in globals() and "y_test" in globals() and "y_pred" in globals():
    cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=clf.classes_
    )

    fig, ax = plt.subplots(figsize=(12, 8))
    disp.plot(ax=ax, xticks_rotation=45)
    plt.title("Confusion Matrix - Job Category Prediction")
    plt.tight_layout()
    plt.show()
else:
    print("Model not trained yet. Run the ML model cell first.")

In [ ]:
# =========================
# 9.2 Cross Validation
# =========================

from sklearn.model_selection import cross_val_score

if "clf" in globals() and "vectorizer" in globals() and len(model_data) >= 20:
    X_all_vec = vectorizer.fit_transform(model_data["clean_text"])
    y_all = model_data["job_category"]

    cv_scores = cross_val_score(
        LogisticRegression(max_iter=1000),
        X_all_vec,
        y_all,
        cv=5,
        scoring="accuracy"
    )

    print("Cross Validation Scores:", cv_scores)
    print("Mean Accuracy:", cv_scores.mean())
    print("Standard Deviation:", cv_scores.std())
else:
    print("Not enough data for cross validation.")

In [ ]:
# =========================
# 9.3 Compare Multiple Models
# =========================

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

if "X_train_vec" in globals() and "X_test_vec" in globals():
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Naive Bayes": MultinomialNB(),
        "Linear SVM": LinearSVC(),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
    }

    results = []

    for model_name, model in models.items():
        model.fit(X_train_vec, y_train)
        predictions = model.predict(X_test_vec)

        results.append({
            "model": model_name,
            "accuracy": accuracy_score(y_test, predictions),
            "precision_macro": precision_score(y_test, predictions, average="macro", zero_division=0),
            "recall_macro": recall_score(y_test, predictions, average="macro", zero_division=0),
            "f1_macro": f1_score(y_test, predictions, average="macro", zero_division=0)
        })

    model_results = pd.DataFrame(results).sort_values("f1_macro", ascending=False)
    display(model_results)

    plt.figure(figsize=(10, 6))
    plot_results = model_results.sort_values("f1_macro")
    plt.barh(plot_results["model"], plot_results["f1_macro"])
    plt.title("Model Comparison by F1 Score")
    plt.xlabel("F1 Score")
    plt.ylabel("Model")
    plt.tight_layout()
    plt.show()
else:
    print("Train/test vectors not available. Run the ML model cell first.")

In [ ]:
# =========================
# 9.4 Most Important Keywords Per Category
# =========================

if "clf" in globals() and hasattr(clf, "coef_"):
    feature_names = np.array(vectorizer.get_feature_names_out())

    for i, category in enumerate(clf.classes_):
        top_indices = clf.coef_[i].argsort()[-15:][::-1]
        top_keywords = feature_names[top_indices]

        print("\nCategory:", category)
        print("Important keywords:", ", ".join(top_keywords))
else:
    print("Feature importance available only for Logistic Regression model.")

In [ ]:
# =========================
# 10. Future trend analysis
# =========================

# This section uses dates if available.
# It compares old vs recent job posts and detects rising skills.

jobs_with_dates = jobs.dropna(subset=["posted_date"]).copy()

if len(jobs_with_dates) > 0:
    jobs_with_dates["month"] = jobs_with_dates["posted_date"].dt.to_period("M").astype(str)

    rows = []
    for _, row in jobs_with_dates.iterrows():
        for skill in row["extracted_skills"]:
            rows.append({
                "month": row["month"],
                "skill": skill,
                "job_id": _
            })

    skill_month = pd.DataFrame(rows)

    if len(skill_month) > 0:
        trend_table = skill_month.groupby(["month", "skill"]).size().reset_index(name="count")
        display(trend_table.head())

        # Calculate trend score using linear regression over months
        month_order = sorted(trend_table["month"].unique())
        month_to_index = {m: i for i, m in enumerate(month_order)}
        trend_table["month_index"] = trend_table["month"].map(month_to_index)

        trend_scores = []

        for skill in trend_table["skill"].unique():
            temp = trend_table[trend_table["skill"] == skill].copy()

            if len(temp) >= 2:
                X = temp[["month_index"]]
                y = temp["count"]
                lr = LinearRegression()
                lr.fit(X, y)
                trend_scores.append({
                    "skill": skill,
                    "trend_score": lr.coef_[0],
                    "total_count": temp["count"].sum()
                })

        trends = pd.DataFrame(trend_scores)
        trends = trends.sort_values(["trend_score", "total_count"], ascending=False)

        print("Rising skills:")
        display(trends.head(20))

        print("Declining skills:")
        display(trends.tail(20))

        plt.figure(figsize=(12, 7))
        top_rising = trends.head(15).sort_values("trend_score")
        plt.barh(top_rising["skill"], top_rising["trend_score"])
        plt.title("Top Rising Skills Based on Job Posting Dates")
        plt.xlabel("Trend Score")
        plt.ylabel("Skill")
        plt.tight_layout()
        plt.show()
    else:
        print("No extracted skills found for dated jobs.")
else:
    print("No date column available. Future trend analysis needs job posting dates.")

In [ ]:
# =========================
# 11. Skill gap analyzer
# =========================

# Change this with your real skills
my_current_skills = [
    "excel",
    "sql",
    "python"
]

# Choose target role/category
target_category = "Data"

def recommend_skills_for_category(target_category, my_skills, top_n=15):
    my_skills = set([s.lower() for s in my_skills])

    category_jobs = jobs[jobs["job_category"] == target_category]

    counter = Counter()
    for skills in category_jobs["extracted_skills"]:
        counter.update(skills)

    recommendations = []
    for skill, freq in counter.most_common():
        if skill.lower() not in my_skills:
            recommendations.append({
                "recommended_skill": skill,
                "market_frequency": freq
            })

    return pd.DataFrame(recommendations).head(top_n)

recommendations = recommend_skills_for_category(target_category, my_current_skills)

print("Your current skills:", my_current_skills)
print("Target category:", target_category)
print("\nRecommended skills to learn:")
display(recommendations)

In [ ]:
# =========================
# 12. Predict category for a new job description
# =========================

new_job_description = '''
We are looking for a Data Analyst with strong SQL, Python, Power BI,
Excel, statistics, reporting, dashboarding and business analysis skills.
'''

new_skills = extract_skills(new_job_description)
new_category = detect_category(new_skills)

print("Extracted skills:", new_skills)
print("Detected category:", new_category)

# If ML model exists, use it too
if "clf" in globals() and "vectorizer" in globals():
    new_vec = vectorizer.transform([clean_text(new_job_description)])
    ml_prediction = clf.predict(new_vec)[0]
    print("ML predicted category:", ml_prediction)

In [ ]:
# =========================
# 13. Build final summary tables
# =========================

summary = {
    "total_jobs_analyzed": len(jobs),
    "jobs_with_skills": int((jobs["skill_count"] > 0).sum()),
    "unique_skills_detected": len(skill_freq),
    "most_requested_skill": skill_freq.iloc[0]["skill"] if len(skill_freq) else None,
    "most_common_category": category_counts.iloc[0]["job_category"] if len(category_counts) else None
}

summary

In [ ]:
import pickle

# SAVE EVERYTHING IN ONE SINGLE PICKLE FILE
with open("SkillGap.pickle", "wb") as f:
    pickle.dump({
        "model": mlp_combined,
        "tfidf": tfidf,
        "scaler": scaler,
        "label_encoder": le,
        "classes": CLASSES
    }, f)

print("✅ SkillGap.pickle saved successfully")

## Conclusion

This notebook gives you:

- A skills extraction system
- Job category detection
- Market demand analysis
- Future skill trend analysis
- Skill gap recommendations
- A basic ML classifier for job categories

For a final-year or portfolio project, you can present it as:

**AI-Based Job Market Skill Gap and Future Trend Analysis System**